In [1]:
from pathlib import Path

import pandas as pd
import psycopg

In [2]:
file_path = Path("../input/Y9999_20260813.xlsx")

df = pd.read_excel(
    file_path,
    header=1
)

df.head()

,年月日,開盤價(元),最高價(元),最低價(元),收盤價(元),成交量(千股),成交值(千元),報酬率％,週轉率％,流通在外股數(千股),...,現金股利率,股價漲跌(元),高低價差%,次日開盤參考價,次日漲停價,次日跌停價,注意股票(A),處置股票(D),全額交割(Y),市場別
0,2026/08/12,45175.70,45529.48,45175.70,45518.07,9972596,892659784,0.8806,0.6156,798784627,...,1.6311,397.35,0.7841,NaN,NaN,NaN,NaN,NaN,NaN,TSE
1,2026/08/11,44883.90,45195.41,44652.06,45120.72,9740372,902052207,0.4273,0.6422,798787184,...,1.6455,191.96,1.2094,NaN,NaN,NaN,NaN,NaN,NaN,TSE
2,2026/08/10,44540.71,45219.40,44540.71,44928.76,10238975,880440723,1.5892,0.6202,798051817,...,1.6253,702.85,1.5346,NaN,NaN,NaN,NaN,NaN,NaN,TSE
3,2026/08/07,44450.19,44827.13,43941.56,44225.91,9376069,850815189,-0.3847,0.5600,798047934,...,1.6513,-170.79,1.9947,NaN,NaN,NaN,NaN,NaN,NaN,TSE
4,2026/08/06,44487.94,44601.24,44024.32,44396.70,10650583,974054973,-0.4817,0.6247,798048468,...,1.6451,-214.90,1.2932,NaN,NaN,NaN,NaN,NaN,NaN,TSE


In [3]:
column_map = {
    "年月日": "trade_date",
    "開盤價(元)": "open_price",
    "最高價(元)": "high_price",
    "最低價(元)": "low_price",
    "收盤價(元)": "close_price",
    "成交量(千股)": "volume_thousand_shares",
    "成交值(千元)": "turnover_value_thousand",
    "報酬率％": "return_pct",
    "週轉率％": "turnover_rate_pct",
    "流通在外股數(千股)": "shares_outstanding_thousand",
    "市值(百萬元)": "market_cap_million",
    "最後揭示買價": "last_bid_price",
    "最後揭示賣價": "last_ask_price",
    "報酬率-Ln": "log_return",
    "市值比重％": "market_cap_weight_pct",
    "成交值比重％": "turnover_value_weight_pct",
    "成交筆數(筆)": "trade_count",
    "本益比-TSE": "pe_tse",
    "本益比-TEJ": "pe_tej",
    "股價淨值比-TSE": "pb_tse",
    "股價淨值比-TEJ": "pb_tej",
    "漲跌停": "price_limit_status",
    "股價營收比-TEJ": "ps_tej",
    "股利殖利率-TSE": "dividend_yield_tse",
    "現金股利率": "cash_dividend_yield",
    "股價漲跌(元)": "price_change",
    "高低價差%": "high_low_spread_pct",
    "次日開盤參考價": "next_open_reference_price",
    "次日漲停價": "next_limit_up_price",
    "次日跌停價": "next_limit_down_price",
    "注意股票(A)": "is_attention",
    "處置股票(D)": "is_disposition",
    "全額交割(Y)": "is_full_delivery",
    "市場別": "market"
}

df = df.rename(columns=column_map)

df["trade_date"] = pd.to_datetime(df["trade_date"]).dt.date

In [5]:
import psycopg

try:
    conn = psycopg.connect(
        host="localhost",
        port=5432,
        dbname="market_data",
        user="market_data_user",
        password="market_data_dev_password",
        connect_timeout=5
    )

    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                current_database(),
                current_user,
                version();
        """)

        database, user, version = cur.fetchone()

    print("PostgreSQL 連線成功")
    print("Database :", database)
    print("User     :", user)
    print("Version  :", version)

except psycopg.Error as e:
    print("PostgreSQL 連線失敗")
    print(e)


PostgreSQL 連線成功
Database : market_data
User     : market_data_user
Version  : PostgreSQL 18.4 (Debian 18.4-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


In [6]:
with conn.cursor() as cur:
    cur.execute("""
        INSERT INTO instrument (
            code,
            name_zh,
            market,
            instrument_type
        )
        VALUES (%s, %s, %s, %s)
        ON CONFLICT (market, code)
        DO UPDATE SET
            name_zh = EXCLUDED.name_zh,
            instrument_type = EXCLUDED.instrument_type,
            updated_at = NOW()
        RETURNING id;
    """, (
        "Y9999",
        "加權指數",
        "TSE",
        "INDEX"
    ))

    instrument_id = cur.fetchone()[0]

conn.commit()

instrument_id

1

In [7]:
df = df.where(pd.notna(df), None)

In [8]:
market_daily_columns = [
    "trade_date",
    "open_price",
    "high_price",
    "low_price",
    "close_price",
    "volume_thousand_shares",
    "turnover_value_thousand",
    "return_pct",
    "turnover_rate_pct",
    "shares_outstanding_thousand",
    "market_cap_million",
    "last_bid_price",
    "last_ask_price",
    "log_return",
    "market_cap_weight_pct",
    "turnover_value_weight_pct",
    "trade_count",
    "pe_tse",
    "pe_tej",
    "pb_tse",
    "pb_tej",
    "price_limit_status",
    "ps_tej",
    "dividend_yield_tse",
    "cash_dividend_yield",
    "price_change",
    "high_low_spread_pct",
    "next_open_reference_price",
    "next_limit_up_price",
    "next_limit_down_price",
    "is_attention",
    "is_disposition",
    "is_full_delivery"
]

In [9]:
sql = """
INSERT INTO market_daily (
    instrument_id,
    trade_date,
    open_price,
    high_price,
    low_price,
    close_price,
    volume_thousand_shares,
    turnover_value_thousand,
    return_pct,
    turnover_rate_pct,
    shares_outstanding_thousand,
    market_cap_million,
    last_bid_price,
    last_ask_price,
    log_return,
    market_cap_weight_pct,
    turnover_value_weight_pct,
    trade_count,
    pe_tse,
    pe_tej,
    pb_tse,
    pb_tej,
    price_limit_status,
    ps_tej,
    dividend_yield_tse,
    cash_dividend_yield,
    price_change,
    high_low_spread_pct,
    next_open_reference_price,
    next_limit_up_price,
    next_limit_down_price,
    is_attention,
    is_disposition,
    is_full_delivery
)
VALUES (
    %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
    %s, %s, %s, %s
)
ON CONFLICT (instrument_id, trade_date)
DO UPDATE SET
    open_price = EXCLUDED.open_price,
    high_price = EXCLUDED.high_price,
    low_price = EXCLUDED.low_price,
    close_price = EXCLUDED.close_price,
    volume_thousand_shares = EXCLUDED.volume_thousand_shares,
    turnover_value_thousand = EXCLUDED.turnover_value_thousand,
    return_pct = EXCLUDED.return_pct,
    turnover_rate_pct = EXCLUDED.turnover_rate_pct,
    shares_outstanding_thousand = EXCLUDED.shares_outstanding_thousand,
    market_cap_million = EXCLUDED.market_cap_million,
    last_bid_price = EXCLUDED.last_bid_price,
    last_ask_price = EXCLUDED.last_ask_price,
    log_return = EXCLUDED.log_return,
    market_cap_weight_pct = EXCLUDED.market_cap_weight_pct,
    turnover_value_weight_pct = EXCLUDED.turnover_value_weight_pct,
    trade_count = EXCLUDED.trade_count,
    pe_tse = EXCLUDED.pe_tse,
    pe_tej = EXCLUDED.pe_tej,
    pb_tse = EXCLUDED.pb_tse,
    pb_tej = EXCLUDED.pb_tej,
    price_limit_status = EXCLUDED.price_limit_status,
    ps_tej = EXCLUDED.ps_tej,
    dividend_yield_tse = EXCLUDED.dividend_yield_tse,
    cash_dividend_yield = EXCLUDED.cash_dividend_yield,
    price_change = EXCLUDED.price_change,
    high_low_spread_pct = EXCLUDED.high_low_spread_pct,
    next_open_reference_price = EXCLUDED.next_open_reference_price,
    next_limit_up_price = EXCLUDED.next_limit_up_price,
    next_limit_down_price = EXCLUDED.next_limit_down_price,
    is_attention = EXCLUDED.is_attention,
    is_disposition = EXCLUDED.is_disposition,
    is_full_delivery = EXCLUDED.is_full_delivery,
    updated_at = NOW();
"""

In [10]:
try:
    with conn.cursor() as cur:
        for _, row in df.iterrows():
            values = [instrument_id]

            for col in market_daily_columns:
                value = row[col]

                if pd.isna(value):
                    value = None

                values.append(value)

            cur.execute(sql, values)

    conn.commit()

    print(f"匯入完成，共處理 {len(df)} 筆資料")

except Exception as e:
    conn.rollback()
    print("匯入失敗")
    print(e)

匯入完成，共處理 6733 筆資料
